# Binomial Options Pricing Model

In [2]:
import math
import matplotlib.pyplot as plt

S0 = 100           # stock price today
K = 100            # strike price
T = 1              # time to expiry (years)
r = 0.05           # risk-free interest rate
sigma = 0.2        # volatility (percent)

# The Payoff
At expirey, an option is worth what you'd get by exercising it.
A call lets you buy at the strike, so it only pays if the stock ends above $K$:
$$C = \max(K - S, \ 0)$$
A put lets you sell at the strike, so it pays only if the stock ends below $K$:
$$P = \max(K - S,\ 0)$$

In [3]:
def payoff(S, K, option_type):
    if option_type == "call":
        return max(S - K, 0)
    else:
        return max(K - S, 0)

## Ups and down moves

The tree splits the time to expiry to $N$ equal steps. At each step the stock either goes up by a factor $u$ or down by a factor $d$:

$$\Delta t = \frac{T}{N}, \qquad u = e^{\sigma\sqrt{\Delta t}}, \qquad d = \frac{1}{u}$$

Since $d = 1/u$, an up move followed by a down move brings the stock back to where it started.

To price the option, we use the probability $p$ of an up move that makes the stock grow at the risk-free rate on average:

$$p = \frac{e^{r\Delta t} - d}{u - d}$$

In [9]:
def tree_parameters(T, r, sigma, N):
    dt = T / N
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    p = (math.exp(r * dt) - d) / (u - d)
    return dt, u, d, p

## The stock price tree
After $i$ steps with $j$ up moves (and $i - j$ down moves), the stock price is: $$S_{i,j} = S_0 \, u^{j} \, d^{\,i-j}$$ We can store the tree as a list of rows. Row $i$ holds the $i + 1$ possible prices after $i$ steps, so `stock[i][j]` is the price after $i$ steps with $j$ up moves.

In [14]:
def stock_tree(S0, u, d, N):
    stock = []
    for i in range(N + 1):
        row = []
        for j in range(i + 1):
            row.append(S0 * u**j * d**(i - j))
        stock.append(row)
    return stock

# Working Backward to today's price
At expiry, the option is worth its payoff. One step earlier, each node can move to an up node or a down node. Its value is the average of those two, weighted by $p$, and discounted back one step (to account for the time value of money) at the risk-free rate:

$$V_{\text{hold}} = e^{-r\Delta t}\left[\,p\,V_{\text{up}} + (1-p)\,V_{\text{down}}\,\right]$$

Repeating this back to step 0 gives the option's price today.

A **European** option can only be exercised at expiry, so its value is just $V_{\text{hold}}$.

An **American** option can be exercised at any step, so at each node it's worth whichever is bigger, holding or exercising now, in addition, because of its flexibility, the American option is usually equal to or worth more then its european counterpart

In [16]:
def option_tree(S0, K, T, r, sigma, N, option_type, american):
    dt, u, d, p = tree_parameters(T, r, sigma, N)
    stock = stock_tree(S0, u, d, N)
    disc = math.exp(-r * dt)
   
    option = []
    exercise = []
    for i in range(N + 1):
        option.append([0] * (i + 1))
        exercise.append([False] * (i + 1))
    for j in range(N + 1):
        option[N][j] = payoff(stock[N][j], K, option_type)

    for i in range(N - 1, -1, -1):
        for j in range(i + 1):
            hold = disc * (p * option[i + 1][j + 1] + (1 - p) * option[i + 1][j])
            now = payoff(stock[i][j], K, option_type)
            if american and now > hold:
                option[i][j] = now
                exercise[i][j] = True
            else:
                option[i][j] = hold

    return option, exercise


def option_price(S0, K, T, r, sigma, N, option_type, american):
    option, exercise = option_tree(S0, K, T, r, sigma, N, option_type, american)
    return option[0][0]